In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from google.colab import drive
from sklearn.preprocessing import PolynomialFeatures, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import statsmodels.api as sm

# Constantes e configurações
DRIVE_MOUNT_PATH = "/content/drive"
GRAPHPATH = "/content/drive/MyDrive/graficos"
SUMMARYPATH = "/content/drive/MyDrive/sumarios"
CSV_URL = "https://docs.google.com/spreadsheets/d/1GAYrQKB-DF7ea7g0PhxRJN_ItxT-n6eGw7N8QEjUZFg/export?format=csv&gid=1955286066"
DEFAULT_PALETTE = "muted"
FIGSIZE = (10, 6)

sns.set(style="whitegrid")
plt.rcParams.update({
    'figure.facecolor': 'black',
    'axes.facecolor': 'black',
    'axes.edgecolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'text.color': 'white',
    'legend.facecolor': 'black',
    'axes.labelcolor': 'white'
})

def load_data(url):
    try:
        df = pd.read_csv(url, skiprows=1, names=['Idade', 'Modelo', 'Sexo'])
        return df
    except Exception as e:
        print(f"Erro ao carregar os dados: {e}")
        return None

def preprocess_numeric_cols(df, cols):
    df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
    return df.dropna(subset=cols)

def save_fig(filename):
    plt.savefig(os.path.join(GRAPHPATH, filename), dpi=300, bbox_inches='tight')
    plt.close()

def create_figure():
    return plt.figure(figsize=FIGSIZE)

# Funções para gerar gráficos de violino
def generate_violinplot(df, x, y, hue, title, filename, split=False):
    create_figure()
    sns.violinplot(x=x, y=y, hue=hue, data=df, split=split, palette=DEFAULT_PALETTE)
    plt.title(title)
    save_fig(filename)

# Funções para gerar histogramas
def generate_histplot(df, x, hue, title, filename, bins=20):
    create_figure()
    sns.histplot(data=df, x=x, hue=hue, bins=bins, palette=DEFAULT_PALETTE)
    plt.title(title)
    save_fig(filename)

# Funções para gerar KDEs
def generate_kdeplot(df, x, hue, title, filename):
    create_figure()
    sns.kdeplot(data=df, x=x, hue=hue, palette=DEFAULT_PALETTE)
    plt.title(title)
    save_fig(filename)

# Funções para gerar gráficos de regressões não lineares
def generate_polynomial_regression(df, x, y, degree, title, filename):
    df = preprocess_numeric_cols(df.copy(), [x, y])
    x_data = df[x].values.reshape(-1, 1)
    y_data = df[y].values.reshape(-1, 1)

    polynomial_features = PolynomialFeatures(degree=degree)
    x_poly = polynomial_features.fit_transform(x_data)

    model = LinearRegression()
    model.fit(x_poly, y_data)

    y_poly_pred = model.predict(x_poly)

    plt.scatter(x_data, y_data, color='blue')
    plt.plot(x_data, y_poly_pred, color='red')
    plt.title(f"{title} (Degree {degree})")
    save_fig(filename)

def generate_logarithmic_regression(df, x, y, title, filename):
    df = preprocess_numeric_cols(df.copy(), [x, y])
    df = df[df[x] > 0]  # Filtra valores de x > 0 para evitar log de 0 ou negativo
    x_data = np.log(df[x]).values.reshape(-1, 1)
    y_data = df[y].values.reshape(-1, 1)

    model = LinearRegression()
    model.fit(x_data, y_data)

    y_log_pred = model.predict(x_data)

    plt.scatter(df[x], y_data, color='blue')
    plt.plot(df[x], y_log_pred, color='red')
    plt.title(title)
    save_fig(filename)

def generate_exponential_regression(df, x, y, title, filename):
    df = preprocess_numeric_cols(df.copy(), [x, y])
    df = df[df[y] > 0]  # Filtra valores de y > 0 para evitar log de 0 ou negativo
    x_data = df[x].values.reshape(-1, 1)
    y_data = np.log(df[y]).values.reshape(-1, 1)

    model = LinearRegression()
    model.fit(x_data, y_data)

    y_exp_pred = np.exp(model.predict(x_data))

    plt.scatter(x_data, df[y], color='blue')
    plt.plot(x_data, y_exp_pred, color='red')
    plt.title(title)
    save_fig(filename)

# Função para gerar sumário estatístico detalhado com intervalo de confiança e valor de p
def generate_statistical_summary_detailed(df, y_var, filename="sumario_detalhado.txt"):
    try:
        X = df[['Idade', 'Sexo_Num', 'Modelo_Num']]
        X = sm.add_constant(X)
        y = df[y_var]

        model = sm.OLS(y, X).fit()
        summary = model.summary()

        with open(os.path.join(SUMMARYPATH, filename), 'w') as f:
            f.write(summary.as_text())
    except Exception as e:
        print(f"Erro ao gerar o sumário estatístico detalhado: {e}")

if __name__ == "__main__":
    drive.mount(DRIVE_MOUNT_PATH, force_remount=True)
    os.makedirs(GRAPHPATH, exist_ok=True)
    os.makedirs(SUMMARYPATH, exist_ok=True)

    df = load_data(CSV_URL)

    if df is not None:
        # Transformar colunas categóricas para numéricas
        le_sexo = LabelEncoder()
        df['Sexo_Num'] = le_sexo.fit_transform(df['Sexo'])

        le_modelo = LabelEncoder()
        df['Modelo_Num'] = le_modelo.fit_transform(df['Modelo'])

        # Gráficos de violino bipartido e não bipartido
        generate_violinplot(df, 'Modelo', 'Idade', 'Sexo', 'Violin: Idade por Sexo e Modelo', 'violin_idade_modelo_sexo.png', split=True)
        generate_violinplot(df, 'Sexo', 'Idade', 'Modelo', 'Violin: Idade por Modelo e Sexo', 'violin_idade_sexo_modelo.png', split=True)
        generate_violinplot(df, 'Modelo', 'Idade', 'Sexo', 'Violin: Idade por Modelo e Sexo', 'violin_idade_modelo_sexo_2.png', split=False)
        generate_violinplot(df, 'Sexo', 'Idade', 'Modelo', 'Violin: Idade por Sexo e Modelo', 'violin_idade_sexo_modelo_2.png', split=False)
        generate_violinplot(df, 'Modelo', 'Idade', 'Sexo', 'Violin: Idade por Sexo e Modelo 3', 'violin_idade_modelo_sexo_3.png', split=True)

        # Histogramas
        generate_histplot(df, 'Idade', 'Sexo', 'Histograma: Idade por Sexo', 'hist_idade_sexo.png', bins=20)
        generate_histplot(df, 'Idade', None, 'Histograma: Idade Geral', 'hist_idade_geral.png', bins=20)

        # KDEs
        generate_kdeplot(df, 'Idade', 'Modelo', 'KDE: Idade por Modelo', 'kde_idade_modelo.png')
        generate_kdeplot(df, 'Idade', 'Sexo', 'KDE: Idade por Sexo', 'kde_idade_sexo.png')

        # Gráficos de regressões não lineares
        generate_polynomial_regression(df, 'Idade', 'Sexo_Num', 2, 'Regressão Polinomial (Grau 2)', 'polynomial_degree_2.png')
        generate_polynomial_regression(df, 'Idade', 'Sexo_Num', 3, 'Regressão Polinomial (Grau 3)', 'polynomial_degree_3.png')
        generate_polynomial_regression(df, 'Idade', 'Modelo_Num', 3, 'Regressão Polinomial (Grau 3)', 'polynomial_model_degree_3.png')
        generate_logarithmic_regression(df, 'Idade', 'Sexo_Num', 'Regressão Logarítmica', 'logarithmic_regression.png')
        generate_logarithmic_regression(df, 'Idade', 'Modelo_Num', 'Regressão Logarítmica Modelo', 'logarithmic_model_regression.png')

        # Sumário estatístico detalhado
        generate_statistical_summary_detailed(df, 'Idade')

Mounted at /content/drive


<ipython-input-26-5dbd4b37edad>:61: UserWarning: Ignoring `palette` because no `hue` variable has been assigned.
  sns.histplot(data=df, x=x, hue=hue, bins=bins, palette=DEFAULT_PALETTE)
